# Data Cleaning Execution

Retain customers and distinguish unknown accumulated charges from zero. Saved outputs are produced by actual sequential Python execution using `python3 scripts/run_audit_notebook.py --stage cleaning`, not by a Jupyter kernel. These cells can also run in order in a Python notebook editor. No modeling or feature-relationship analysis is performed.

## 1. Inspect the transformation and execute cleaning

The printed function is the same function imported by the cleaning pipeline. Capture the call, function body and resulting counts together.


In [1]:
from pathlib import Path
import csv, hashlib, inspect, json, subprocess, sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/raw/source_manifest.json").exists())
sys.path.insert(0, str(ROOT / "scripts"))
from clean_data import normalize_total_charges

print(inspect.getsource(normalize_total_charges))
run = subprocess.run([sys.executable, str(ROOT / "scripts/clean_data.py")],
                     cwd=ROOT, capture_output=True, text=True, check=True)
print(run.stdout, end="")


def normalize_total_charges(raw_value):
    """Represent unknown charges explicitly without imputing a numeric value."""
    is_missing = raw_value.strip() == ''
    return ('' if is_missing else raw_value), ('1' if is_missing else '0')

CLEANING COMPLETE | 7,043 -> 7,043 rows | 21 -> 22 fields
Original cells changed: 11 | Deleted: 0 | Imputed: 0
TotalChargesMissing: 1=11, 0=7032
Verified: raw preserved, all other cells unchanged, CSV round-trip, target counts unchanged.


## 2. Compare original and cleaned records

Load the two files independently and display every record whose original TotalCharges field was blank. The repr output makes the difference between a single space and an empty missing field visible. CSV has no stored numeric types; the loading contract is in `data/processed/schema.json`.


In [2]:
manifest = json.loads((ROOT / "data/raw/source_manifest.json").read_text())
raw_path = ROOT / "data/raw" / manifest["file"]
with raw_path.open(newline="") as stream:
    raw = list(csv.DictReader(stream))
with (ROOT / "data/processed/telco_clean.csv").open(newline="") as stream:
    cleaned = list(csv.DictReader(stream))
by_id = {r["customerID"]: r for r in cleaned}
print("customerID    before  after  missing_flag  tenure")
for r in raw:
    if r["TotalCharges"].strip() == "":
        c = by_id[r["customerID"]]
        print(f"{r['customerID']:12} {r['TotalCharges']!r:7} {c['TotalCharges']!r:6}"
              f" {c['TotalChargesMissing']:>12} {c['tenure']:>7}")
assert [r["customerID"] for r in raw] == [r["customerID"] for r in cleaned]
print(f"All {len(raw):,} customer IDs and their row order are retained.")


customerID    before  after  missing_flag  tenure
4472-LVYGI   ' '     ''                1       0
3115-CZMZD   ' '     ''                1       0
5709-LVOEQ   ' '     ''                1       0
4367-NUYAO   ' '     ''                1       0
1371-DWPAZ   ' '     ''                1       0
7644-OMVMY   ' '     ''                1       0
3213-VVOLG   ' '     ''                1       0
2520-SGTTA   ' '     ''                1       0
2923-ARZLG   ' '     ''                1       0
4075-WKNIU   ' '     ''                1       0
2775-SEFEE   ' '     ''                1       0
All 7,043 customer IDs and their row order are retained.


## 3. Verify preservation and edge cases

Compare all original cells, not just selected examples. Then run the independent verification script, which checks real-zero preservation, target-independent missingness, rejection of invalid inputs and byte-for-byte reproducibility.


In [3]:
changes = [(before["customerID"], field, before[field], after[field])
           for before, after in zip(raw, cleaned)
           for field in before if before[field] != after[field]]
assert len(changes) == 11
assert all(field == "TotalCharges" and old == " " and new == ""
           for _, field, old, new in changes)
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == manifest["sha256"]
assert normalize_total_charges("0") == ("0", "0")
assert normalize_total_charges(" ") == ("", "1")
print("Exactly 11 original cells changed; all other original values are unchanged.")
print("Verified: numeric zero stays zero; whitespace becomes explicitly missing.")
check = subprocess.run([sys.executable, str(ROOT / "scripts/verify_cleaning.py")],
                       capture_output=True, text=True, check=True)
print(check.stdout, end="")


Exactly 11 original cells changed; all other original values are unchanged.
Verified: numeric zero stays zero; whitespace becomes explicitly missing.
CLEANING COMPLETE | 7,043 -> 7,043 rows | 21 -> 22 fields
Original cells changed: 11 | Deleted: 0 | Imputed: 0
TotalChargesMissing: 1=11, 0=7032
Verified: raw preserved, all other cells unchanged, CSV round-trip, target counts unchanged.
VERIFIED: exactly 11 original cells changed; all 7,043 customers retained.
VERIFIED: zero is preserved, missingness is target-independent, invalid inputs fail.
VERIFIED: 5 generated files reproduce byte-for-byte from outside the repository root.


## Handoff boundary

The cleaned CSV has 7,043 rows and 22 fields. TotalCharges still has 11 unknown values. TotalChargesMissing is a data-quality indicator; whether to use it as a predictor is a modeling choice. C still needs to choose the train/test split, model-side imputation and encoding. The decision log records why records were retained rather than deleted or filled with zero.
